In [34]:
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from importlib import reload
import data_loading  # Import the module instead of specific functions


reload(data_loading)  # Reload the module after making changes

# Now you can reference the functions directly from the reloaded module
read_csv_to_dataframe = data_loading.read_csv_to_dataframe
read_txt_to_dataframe = data_loading.read_txt_to_dataframe

In [35]:
## making file path imports more robust

# get directory of current file
current_script_directory = Path.cwd()

# Construct path to data files given relative location
usa_string = current_script_directory  / "../data/raw/usa/"
usa_jan2002_dec2009 = usa_string / "accident_hazardous_liquid_jan2002_dec2009/accident_hazardous_liquid_jan2002_dec2009.txt"

# read data into dataframe
usa_jan2002_dec2009_raw = read_txt_to_dataframe(usa_jan2002_dec2009)

TXT file at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/usa/accident_hazardous_liquid_jan2002_dec2009/accident_hazardous_liquid_jan2002_dec2009.txt' successfully read into a DataFrame.


# Cleaning

In [36]:
clean_df = usa_jan2002_dec2009_raw.copy()

In [37]:
## focus on class_txt = "Crude Oil"
## class_txt is the main classifier, but comm provides the detailed info
clean_df = clean_df.loc[clean_df['class_txt']=="CRUDE OIL"]

# we really care about the details in the comm so once we've filtered for crude oil let's drop comm=na as they don't provide us with additional value (and cause problems later on)
clean_df = clean_df.loc[clean_df['comm'].isna()==False]

## let's map strings to binary columns; specifically, heavy, medium, light, sour, and sweet
## other mappings: HLS - heavy louisana sweet (included in if statement)
new_cols = ['heavy', 'medium', 'light', 'sour', 'sweet']
for col in new_cols:
    clean_df[col] = 0
    if ((col == 'heavy') | (col == 'sweet')):
        clean_df.loc[(clean_df['comm'].str.contains(col.upper())) | (clean_df['comm'].str.contains('HLS')), col]=1
    else:
        clean_df.loc[(clean_df['comm'].str.contains(col.upper())), col]=1

## BRIEF - drop any rows that don't have a classification from above
clean_df = clean_df[clean_df[new_cols].sum(axis=1) != 0]



In [38]:
## DESCRIPTIVE ANALYSIS
# check for incident counts by characteristic
clean_df[['heavy', 'medium', 'light', 'sour', 'sweet']].sum()

## results show that sweet make up larger proportion than other characteristics, although it is still a small portion of the total (>1k)

heavy     10
medium     2
light      7
sour      11
sweet     25
dtype: int64

In [39]:
#### two analyses based on spill amount
## IF spill is less than 5 barrels, use gen_cause/gen_cause_txt (options 1-8)
## elseif spill >5 barrels, use cause/cause_txt (options 1-25)
## REF - 42 gallons in a barrel

## IN GENERAL, the gen_cause and cause are not correctly filled in - people often filled in cause when they're not supposed to
## therefore, let's take the more accurate classifiction - gen cause - and if it doesn't exist, THEN take cause
## let's not worry about the size of the loss just yet
test_df = clean_df.copy()
test_df = test_df[['heavy', 'medium', 'light', 'sour', 'sweet', 'gen_cause', 'cause']]
test_df['final_cause'] = test_df['gen_cause'].astype(int)
#test_df['final_cause'] = test_df['cause'].fillna(test_df['gen_cause'])

## originally used more specific categories but this left sample sizes to be too small; trying again with gen_cause

# no na's so drop old cause cols
test_df.drop(columns=['cause', 'gen_cause'], inplace=True)


In [40]:
test_df.groupby(['final_cause']).sum()

,heavy,medium,light,sour,sweet
final_cause,,,,,
1,1,2,1,3,11
2,1,0,0,1,1
3,0,0,0,0,1
5,5,0,0,0,4
6,2,0,4,5,3
7,1,0,1,2,3
8,0,0,1,0,2


In [41]:
## BRIEF - subtract 1 from final_cause col
test_df['final_cause'] = test_df['final_cause']-1

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pandas as pd

# Assuming test_df is your DataFrame
X = test_df[['heavy', 'medium', 'light', 'sour', 'sweet']]  # Independent variables
y = test_df['final_cause']  # Dependent variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit the logistic regression model
model = LogisticRegression(multi_class='multinomial', solver='lbfgs')
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         3
           1       0.00      0.00      0.00         1
           4       0.50      1.00      0.67         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1

    accuracy                           0.44         9
   macro avg       0.17      0.33      0.22         9
weighted avg       0.22      0.44      0.30         9



/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is il

In [43]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Assuming test_df is already defined and preprocessed
X = test_df.drop('final_cause', axis=1)  # or 'final_cause_reduced' if using reduced categories
y = test_df['final_cause']  # or 'final_cause_reduced'

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
## STEP 1 - split clean_df based on spill amount
## NOTE: they filled in GEN_CAUSE regardless of whether it was a small spill or big spill
## the gen_cause will be redundant for these and we'd rather be specific so only use big spill

## LOGIC
# used gallons and 

In [70]:
# test_df = clean_df.groupby(['comm', 'class_txt'])[['heavy', 'medium', 'light', 'sour', 'sweet']].sum().reset_index()
# test_df.to_csv('classifications.csv', index=False)


In [ ]:
## POTENTIALLY relevant columns
# Cause for small spills -  GEN_CAUSE, GEN_CAUSE_TXT
# where failure occured - FAIL_OC, FAIL_OC_TXT; year component was installed - PRTYR; overpressurization - OPRS

In [71]:
## we can now drop comm and class_txt columns because their characteristics are included in the other 5 cols
test_df

,comm,class_txt,heavy,medium,light,sour,sweet
0,CRUDE OIL,CRUDE OIL,0,0,0,0,0
1,"""CRUDE OIL, COLD LAKE""",CRUDE OIL,0,0,0,0,0
2,ALASKA NORTH SLOPE CRUDE,CRUDE OIL,0,0,0,0,0
3,ALASKA NORTH SLOPE CRUDE OIL,CRUDE OIL,0,0,0,0,0
4,ALBIAN HEAVY SYNTHETIC,CRUDE OIL,1,0,0,0,0
...,...,...,...,...,...,...,...
96,WESTERN CANADIAN SELECT,CRUDE OIL,0,0,0,0,0
97,WILMINGTON CRUDE,CRUDE OIL,0,0,0,0,0
98,WTI DOMESTIC SWEET TYPE CRUDE OIL,CRUDE OIL,0,0,0,0,2
99,WX (SOUR CRUDE),CRUDE OIL,0,0,0,1,0


In [14]:
## looking for substance carried
clean_df.loc[clean_df['productcarriedid_displayeng']!=clean_df['productreleaseddisplayeng']][['productcarriedid_displayeng', 'productreleaseddisplayeng']].drop_duplicates()

## based on the above we can just use productcarriedid_displayeng. let's also drop any rows where it is n/a as they are not useful.
clean_df = clean_df.loc[clean_df['productcarriedid_displayeng'].isna()==False]

In [16]:

clean_df.groupby(['productcarriedid_displayeng']).count()

## ONLY 2 ROWS where we have sour. let's move on to another dataset. 
clean_df.loc[clean_df['productcarriedid_displayeng']=='Crude Oil-Sour']


,occno,occid,occclassid,occclassid_displayeng,occclassid_displayfre,occtypeid,occtypeid_displayeng,occtypeid_displayfre,occdate,occtime,...,notification_details_provinceid,notification_details_provinceid_displayeng,notification_details_provinceid_displayfre,tsbnotifiednebdate,tsbnotifiednebtime,dateofcall,timeofcall,totalseriousinjuries,totalfatalinjuries,dailyreleaseddate
1272,P13H0011,96047,9,5,5,3,INCIDENT,INCIDENT,2/2/2013 12:00:00 AM,10:30:00,...,NaN,NaN,NaN,2/2/2013 12:00:00 AM,13:10:00,NaN,NaN,0,0,9/25/2017 12:00:00 AM
1571,P15H0098,96407,9,5,5,3,INCIDENT,INCIDENT,10/15/2015 12:00:00 AM,9:15:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,9/25/2017 12:00:00 AM
